# AFASIA — Inteligência Artificial Pictórica (IAP)
  ## Kaggle Notebook — Gemma 4 Good Hackathon 2026
  ### Atlas Topológico da Afasia + Algoritmo JP (Dijkstra Regressivo)

  **Projeto:** Atlas Topológico da Afasia — IAP (Inteligência Artificial Pictórica)  
  **Autor:** João Pedro Pereira Passos (UFT — Universidade Federal do Tocantins, 2024)  
  **Competição:** [Gemma 4 Good Hackathon — Kaggle / Google DeepMind](https://www.kaggle.com/competitions/gemma-4-good)  
  **Repositório:** [github.com/joaopedropassostocantins/AFASIA](https://github.com/joaopedropassostocantins/AFASIA) ← branch `disfasia`

  ---

  ## Teoria IAP e Algoritmo JP (Passos, 2024)

  A **Inteligência Artificial Pictórica (IAP)** propõe uma camada semântica *pré-linguística*: em vez de processar texto, opera diretamente sobre **estruturas topológicas do significado** — representadas como vetores 12D e analisadas com homologia persistente.

  ### Equação Central: $G \approx I + F$

  | Variável | Nome | Descrição |
  |----------|------|-----------|
  | $G$ | **Dinâmica do Conhecimento** | O estado em transformação — o conceito em movimento |
  | $I$ | **Incerteza** | O espaço topológico atual do usuário |
  | $F$ | **Flexibilidade** | A capacidade de encontrar o próximo passo semântico ótimo |

  ### Pipeline deste Notebook (100% Python puro)

  ```
  25 pictogramas AFASIA (palavra + categoria)
      ↓ [Gemma 4 31B via API Google AI Studio]
  Vetores semânticos 12D (dimensões IAP)
      ↓ [Homologia Persistente H0 — numpy puro]
  Diagramas de persistência Vietoris-Rips simplificados
      ↓ [Wasserstein via scipy.optimize.linear_sum_assignment]
  Matriz Wasserstein 25×25 (distância topológica)
      ↓ [sklearn.manifold.MDS]
  Atlas Pictórico da Afasia
      ↓ [AlgoritmoJP — Dijkstra Regressivo]
  Plano semântico ótimo: fome → comer → maçã
  ```

  **Por que é original?** O Algoritmo JP usa Wasserstein topológica (mede *forma*) — aplicado a CAA/afasia com Dijkstra regressivo, inédito na literatura (Passos, 2024).

In [ ]:
# =============================================================================
  # CÉLULA 1 — SETUP & CONFIGURAÇÃO
  # Dependência extra: google-generativeai
  # Tudo mais (numpy, scipy, sklearn, matplotlib, Pillow, requests) já está no Kaggle
  # =============================================================================

  import subprocess, sys
  subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'google-generativeai'])

  import os, re, json, time, math, heapq, unicodedata
  import numpy as np
  import matplotlib.pyplot as plt
  import matplotlib.patches as mpatches
  from collections import Counter
  from dataclasses import dataclass, field
  from typing import List, Dict
  from io import BytesIO
  from scipy.optimize import linear_sum_assignment
  from sklearn.manifold import MDS
  from sklearn.decomposition import PCA
  import google.generativeai as genai

  try:
      from PIL import Image
      import requests
      PIL_OK = True
      print('[OK] PIL + requests disponíveis (imagens ARASAAC serão exibidas)')
  except ImportError:
      PIL_OK = False
      print('[AVISO] PIL/requests indisponível — grid de imagens desativado')

  # ── Kaggle Secret / Env Var ──────────────────────────────────────────────────
  GEMINI_API_KEY = None
  try:
      from kaggle_secrets import UserSecretsClient
      GEMINI_API_KEY = UserSecretsClient().get_secret('GEMINI_API_KEY')
      print('[OK] GEMINI_API_KEY via Kaggle Secrets')
  except Exception:
      GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY')
      if GEMINI_API_KEY:
          print('[OK] GEMINI_API_KEY via variável de ambiente')
      else:
          print('[AVISO] Sem GEMINI_API_KEY — vetores pré-computados (Gemma 4 31B) serão usados.')

  GEMMA_API_ATIVO = bool(GEMINI_API_KEY)
  if GEMMA_API_ATIVO:
      genai.configure(api_key=GEMINI_API_KEY)
      GEMMA_MODEL_NAME = 'gemma-4-31b-it'
      gemma_model = genai.GenerativeModel(GEMMA_MODEL_NAME)
      print(f'[OK] Gemma 4 31B via API ({GEMMA_MODEL_NAME})')
  else:
      gemma_model = None
      GEMMA_MODEL_NAME = 'gemma-4-31b-it (pré-computado)'

  import numpy, scipy, sklearn, matplotlib
  print(f'[OK] numpy {numpy.__version__} | scipy {scipy.__version__} | sklearn {sklearn.__version__}')
  print('[PRONTO] Todas as dependências carregadas.')

In [ ]:
# =============================================================================
  # CÉLULA 2 — ARQUITETURA DO ALGORITMO JP
  # =============================================================================

  print('''
  ╔══════════════════════════════════════════════════════════════════════════════╗
  ║         ALGORITMO JP — INTELIGÊNCIA ARTIFICIAL PICTÓRICA (IAP)              ║
  ║         João Pedro Pereira Passos · UFT · Hackathon Gemma 4 Good 2026       ║
  ╠══════════════════════════════════════════════════════════════════════════════╣
  ║                                                                              ║
  ║  FASE 1 — PRÉ-CAMADA SEMÂNTICA (Gemma 4 31B, executa UMA VEZ por ícone)   ║
  ║    palavra em pt-BR → prompt IAP 12D → Gemma 4 31B → vetor [0..10]^12     ║
  ║    Ex: fome → {urgencia:9, saude_corpo:9, necessidade_basica:10, ...}      ║
  ║    Armazenado em cache permanente — sem custo recorrente de inferência      ║
  ║                                                                              ║
  ║  FASE 2 — INTELIGÊNCIA FLUIDA (Python puro, scipy, sklearn)                ║
  ║    vetor 12D → diagrama H0 (Vietoris-Rips 1D) → Wasserstein → MDS         ║
  ║                                                                              ║
  ║  FASE 3 — ALGORITMO JP: PLANEJAMENTO REGRESSIVO (Dijkstra)                 ║
  ║    Grafo Wasserstein → Dijkstra(fome → maçã) → Plano topológico ótimo      ║
  ║    G ≈ I + F: G=caminho | I=estado atual (fome) | F=3 vizinhos de menor    ║
  ║    custo topológico = os próximos pictogramas mais semanticamente próximos  ║
  ║                                                                              ║
  ╠══════════════════════════════════════════════════════════════════════════════╣
  ║  POR QUE WASSERSTEIN SUPERA COSSENO?                                        ║
  ║    Cosseno: mede DIREÇÃO (ângulo entre vetores)                             ║
  ║    Wasserstein: mede FORMA (estrutura de eventos topológicos)               ║
  ║    Dijkstra sobre Wasserstein: caminho SEMANTICAMENTE ótimo — não linear   ║
  ╚══════════════════════════════════════════════════════════════════════════════╝
  ''')
  

In [ ]:
# =============================================================================
  # CÉLULA 3 — 12 DIMENSÕES IAP + 25 PICTOGRAMAS AFASIA + VETORES PRÉ-COMPUTADOS
  # =============================================================================

  IAP_DIMENSOES = [
      'concretude',        # 0:  quão concreto/tangível (10=muito concreto, 0=abstrato)
      'emocionalidade',    # 1:  carga emocional (10=muito emocional, 0=neutro)
      'acao',              # 2:  ação/movimento (10=ação intensa, 0=estático)
      'social',            # 3:  interação social (10=altamente social, 0=isolado)
      'urgencia',          # 4:  urgência/prioridade (10=urgente, 0=rotineiro)
      'temporalidade',     # 5:  relação com tempo (10=muito temporal, 0=atemporal)
      'localizacao',       # 6:  espaço/lugar (10=espacial, 0=sem localização)
      'saude_corpo',       # 7:  saúde/corpo (10=muito corporal, 0=abstrato)
      'comunicacao',       # 8:  comunicação (10=central para CAA, 0=não comunicativo)
      'cognitivo',         # 9:  complexidade cognitiva (10=complexo, 0=simples)
      'necessidade_basica',# 10: necessidade básica (10=fundamental, 0=opcional)
      'especializado_caa', # 11: específico de CAA/AAC (10=vocabulário central CAA, 0=genérico)
  ]
  N_DIM = len(IAP_DIMENSOES)  # 12

  # Vetores pré-computados pelo Gemma 4 31B | formato: {palavra: [d0..d11]}
  PRECOMPUTED_VECTORS = {
      'agua':     [9, 2, 1, 2, 7, 2, 1, 8, 9, 2, 10, 10],
      'comida':   [9, 3, 3, 4, 7, 2, 1, 8, 9, 2, 10, 10],
      'fome':     [3, 7, 1, 1, 9, 2, 1, 9, 9, 2, 10, 10],
      'sede':     [3, 6, 1, 1, 9, 2, 1, 9, 9, 2, 10, 10],
      'remedio':  [8, 4, 2, 4, 8, 6, 6, 9, 9, 4,  9,  9],
      'ajuda':    [2, 6, 7, 9, 9, 2, 1, 5, 10, 2, 10, 10],
      'banheiro': [9, 2, 2, 2, 7, 2, 8, 7, 9, 2, 10,  9],
      'dor':      [2, 10, 1, 2, 9, 2, 1, 10, 9, 2,  9,  9],
      'feliz':    [1, 10, 1, 7, 3, 2, 1, 4, 9, 4,  4,  9],
      'triste':   [1, 10, 1, 5, 4, 2, 1, 5, 9, 4,  5,  9],
      'medo':     [1, 10, 1, 2, 8, 2, 1, 7, 9, 4,  6,  8],
      'cansado':  [2, 6, 1, 1, 5, 2, 1, 8, 8, 2,  6,  8],
      'casa':     [9, 5, 1, 7, 2, 2, 10, 2, 8, 2,  8,  8],
      'hospital': [9, 6, 1, 6, 8, 5, 9, 8, 9, 4,  8,  9],
      'escola':   [9, 4, 1, 8, 2, 6, 9, 1, 8, 5,  6,  7],
      'eu':       [1, 5, 1, 9, 2, 2, 1, 3, 10, 2,  7, 10],
      'familia':  [2, 8, 1, 10, 2, 2, 1, 2, 9, 2,  8,  9],
      'medico':   [8, 5, 1, 9, 8, 4, 7, 8, 9, 5,  8,  9],
      'amigo':    [2, 8, 3, 10, 2, 2, 1, 2, 9, 3,  5,  8],
      'quero':    [1, 5, 7, 9, 6, 2, 1, 2, 10, 3,  7, 10],
      'sim':      [1, 4, 2, 9, 5, 2, 1, 1, 10, 2,  6, 10],
      'nao':      [1, 5, 2, 9, 6, 2, 1, 1, 10, 2,  7, 10],
      'comer':    [4, 4, 8, 5, 8, 2, 1, 8, 9, 2, 10,  9],
      'ir':       [1, 2, 9, 5, 4, 6, 7, 2, 9, 2,  5,  8],
      'maca':     [9, 2, 1, 1, 4, 2, 1, 7, 8, 2,  8,  8],
  }

  # 25 Pictogramas AFASIA — IDs ARASAAC reais (CC BY-NC-SA)
  PICTOGRAMAS_IAP = [
      # necessidades (7) — necessidades fisiológicas básicas
      {'id': 2077,  'palavra': 'agua',     'categoria': 'necessidades',
       'url': 'https://static.arasaac.org/pictograms/2077/2077_500.png'},
      {'id': 2948,  'palavra': 'comida',   'categoria': 'necessidades',
       'url': 'https://static.arasaac.org/pictograms/2948/2948_500.png'},
      {'id': 4828,  'palavra': 'fome',     'categoria': 'necessidades',
       'url': 'https://static.arasaac.org/pictograms/4828/4828_500.png'},
      {'id': 24877, 'palavra': 'sede',     'categoria': 'necessidades',
       'url': 'https://static.arasaac.org/pictograms/24877/24877_500.png'},
      {'id': 5889,  'palavra': 'remedio',  'categoria': 'necessidades',
       'url': 'https://static.arasaac.org/pictograms/5889/5889_500.png'},
      {'id': 12252, 'palavra': 'ajuda',    'categoria': 'necessidades',
       'url': 'https://static.arasaac.org/pictograms/12252/12252_500.png'},
      {'id': 2392,  'palavra': 'banheiro', 'categoria': 'necessidades',
       'url': 'https://static.arasaac.org/pictograms/2392/2392_500.png'},
      # sentimentos (5) — estados emocionais
      {'id': 3085,  'palavra': 'dor',      'categoria': 'sentimentos',
       'url': 'https://static.arasaac.org/pictograms/3085/3085_500.png'},
      {'id': 9907,  'palavra': 'feliz',    'categoria': 'sentimentos',
       'url': 'https://static.arasaac.org/pictograms/9907/9907_500.png'},
      {'id': 35545, 'palavra': 'triste',   'categoria': 'sentimentos',
       'url': 'https://static.arasaac.org/pictograms/35545/35545_500.png'},
      {'id': 6054,  'palavra': 'medo',     'categoria': 'sentimentos',
       'url': 'https://static.arasaac.org/pictograms/6054/6054_500.png'},
      {'id': 2805,  'palavra': 'cansado',  'categoria': 'sentimentos',
       'url': 'https://static.arasaac.org/pictograms/2805/2805_500.png'},
      # lugares (3) — locais de referência
      {'id': 4661,  'palavra': 'casa',     'categoria': 'lugares',
       'url': 'https://static.arasaac.org/pictograms/4661/4661_500.png'},
      {'id': 4829,  'palavra': 'hospital', 'categoria': 'lugares',
       'url': 'https://static.arasaac.org/pictograms/4829/4829_500.png'},
      {'id': 4833,  'palavra': 'escola',   'categoria': 'lugares',
       'url': 'https://static.arasaac.org/pictograms/4833/4833_500.png'},
      # pessoas (4) — referências pessoais e sociais
      {'id': 8295,  'palavra': 'eu',       'categoria': 'pessoas',
       'url': 'https://static.arasaac.org/pictograms/8295/8295_500.png'},
      {'id': 4610,  'palavra': 'familia',  'categoria': 'pessoas',
       'url': 'https://static.arasaac.org/pictograms/4610/4610_500.png'},
      {'id': 5338,  'palavra': 'medico',   'categoria': 'pessoas',
       'url': 'https://static.arasaac.org/pictograms/5338/5338_500.png'},
      {'id': 2058,  'palavra': 'amigo',    'categoria': 'pessoas',
       'url': 'https://static.arasaac.org/pictograms/2058/2058_500.png'},
      # acoes (6) — ações e respostas comunicativas
      {'id': 36228, 'palavra': 'quero',    'categoria': 'acoes',
       'url': 'https://static.arasaac.org/pictograms/36228/36228_500.png'},
      {'id': 6929,  'palavra': 'sim',      'categoria': 'acoes',
       'url': 'https://static.arasaac.org/pictograms/6929/6929_500.png'},
      {'id': 6900,  'palavra': 'nao',      'categoria': 'acoes',
       'url': 'https://static.arasaac.org/pictograms/6900/6900_500.png'},
      {'id': 4577,  'palavra': 'comer',    'categoria': 'acoes',
       'url': 'https://static.arasaac.org/pictograms/4577/4577_500.png'},
      {'id': 6560,  'palavra': 'ir',       'categoria': 'acoes',
       'url': 'https://static.arasaac.org/pictograms/6560/6560_500.png'},
      {'id': 2070,  'palavra': 'maca',     'categoria': 'acoes',
       'url': 'https://static.arasaac.org/pictograms/2070/2070_500.png'},
  ]

  CAT_COLORS = {
      'necessidades': '#e74c3c',
      'sentimentos':  '#9b59b6',
      'lugares':      '#27ae60',
      'pessoas':      '#3498db',
      'acoes':        '#f39c12',
  }
  CAT_LABELS = {
      'necessidades': 'Necessidades (7)',
      'sentimentos':  'Sentimentos (5)',
      'lugares':      'Lugares (3)',
      'pessoas':      'Pessoas (4)',
      'acoes':        'Acoes (6)',
  }

  def _norm(s):
      nfd = unicodedata.normalize('NFD', s.lower())
      return ''.join(c for c in nfd if unicodedata.category(c) != 'Mn')

  print(f'[OK] {len(PICTOGRAMAS_IAP)} pictogramas AFASIA + {len(PRECOMPUTED_VECTORS)} vetores pré-computados')
  cnt = Counter(p['categoria'] for p in PICTOGRAMAS_IAP)
  for cat, n in sorted(cnt.items()):
      print(f'  {cat:15s}: {n} pictogramas')
  print(f'  Dimensoes IAP: {N_DIM}')

In [ ]:
# =============================================================================
  # CÉLULA 4 — GRADE DE IMAGENS ARASAAC POR CATEGORIA
  # Baixa os 25 pictogramas via requests e exibe com matplotlib + PIL
  # =============================================================================

  def download_image(url, timeout=12):
      try:
          resp = requests.get(url, timeout=timeout)
          if resp.status_code == 200:
              return Image.open(BytesIO(resp.content)).convert('RGBA')
      except Exception:
          pass
      return None

  def placeholder_image(size=(128, 128)):
      return Image.new('RGBA', size, (60, 60, 60, 255))

  if PIL_OK:
      cats_order = ['necessidades', 'sentimentos', 'lugares', 'pessoas', 'acoes']
      icons_by_cat = {c: [p for p in PICTOGRAMAS_IAP if p['categoria'] == c]
                      for c in cats_order}
      max_n = max(len(v) for v in icons_by_cat.values())

      fig, axes = plt.subplots(len(cats_order), max_n,
                               figsize=(max_n * 1.9, len(cats_order) * 2.4))
      fig.patch.set_facecolor('#1a1a2e')
      print('[IMAGENS] Baixando 25 pictogramas ARASAAC...', flush=True)

      for r, cat in enumerate(cats_order):
          icons = icons_by_cat[cat]
          for c in range(max_n):
              ax = axes[r][c]
              ax.set_facecolor('#1a1a2e')
              ax.axis('off')
              if c < len(icons):
                  ic = icons[c]
                  img = download_image(ic['url']) or placeholder_image()
                  ax.imshow(img)
                  ax.set_title(ic['palavra'], fontsize=7.5, color='white',
                               pad=3, fontweight='bold')
          axes[r][0].set_ylabel(cat.upper(), fontsize=10,
                                color=CAT_COLORS[cat], rotation=90,
                                labelpad=6, fontweight='bold')

      fig.suptitle(
          'Atlas Afasia — 25 Pictogramas ARASAAC por Categoria\n'
          'Fonte: ARASAAC CC BY-NC-SA | Vetores: Gemma 4 31B (IAP, Passos UFT 2024)',
          color='white', fontsize=12, y=1.01
      )
      plt.tight_layout()
      plt.show()
      print('[OK] Grade de imagens exibida.')
  else:
      print('[AVISO] PIL/requests indisponível. Pictogramas por categoria:')
      for cat in ['necessidades', 'sentimentos', 'lugares', 'pessoas', 'acoes']:
          icons = [p for p in PICTOGRAMAS_IAP if p['categoria'] == cat]
          words = ', '.join(p['palavra'] for p in icons)
          print(f'  {cat:14s} ({len(icons)}): {words}')

In [ ]:
# =============================================================================
  # CÉLULA 5 — VETORES SEMÂNTICOS 12D VIA GEMMA 4 31B
  # Modo API:      google-generativeai com prompt IAP 12D
  # Modo fallback: PRECOMPUTED_VECTORS (Gemma 4 31B)
  # =============================================================================

  def IAP_PROMPT(word):
      dims = (
          '1. concretude (10=objeto fisico, 0=abstrato)\n'
          '2. emocionalidade (10=forte emocao, 0=neutro)\n'
          '3. acao (10=acao intensa, 0=estatico)\n'
          '4. social (10=social, 0=isolado)\n'
          '5. urgencia (10=urgente, 0=nao urgente)\n'
          '6. temporalidade (10=temporal, 0=atemporal)\n'
          '7. localizacao (10=localizacao fisica, 0=sem localizacao)\n'
          '8. saude_corpo (10=corporal, 0=nao relacionado)\n'
          '9. comunicacao (10=central para CAA, 0=nao comunicativo)\n'
          '10. cognitivo (10=complexo, 0=simples)\n'
          '11. necessidade_basica (10=fundamental, 0=superfluo)\n'
          '12. especializado_caa (10=vocabulario central CAA, 0=generico)\n'
      )
      return (
          'Especialista em CAA (Comunicacao Aumentativa e Alternativa) e IAP.\n'
          f'Para o conceito: "{word}" (contexto: CAA para afasia em adultos)\n\n'
          'Pontue 0-10 para cada dimensao IAP:\n' + dims +
          'Responda APENAS com JSON valido (sem markdown):\n'
          '{"concretude":X,"emocionalidade":X,"acao":X,"social":X,"urgencia":X,'
          '"temporalidade":X,"localizacao":X,"saude_corpo":X,"comunicacao":X,'
          '"cognitivo":X,"necessidade_basica":X,"especializado_caa":X}'
      )

  def parse_response(text):
      m = re.search(r'\{[\s\S]*?\}', text)
      if m:
          try:
              p = json.loads(m.group())
              v = [float(p.get(d, 5)) for d in IAP_DIMENSOES]
              return [max(0.0, min(10.0, x)) for x in v]
          except Exception:
              pass
      result = {}
      for d in IAP_DIMENSOES:
          m2 = re.search(rf'{d}[:\s*]+(10|[0-9](?:\.[0-9]+)?)', text, re.IGNORECASE)
          if m2:
              result[d] = max(0.0, min(10.0, float(m2.group(1))))
      if len(result) >= 10:
          return [result.get(d, 5.0) for d in IAP_DIMENSOES]
      return None

  def get_vector(picto):
      w = picto['palavra']
      wn = _norm(w)
      norm_map = {_norm(k): k for k in PRECOMPUTED_VECTORS}
      if w in PRECOMPUTED_VECTORS:
          return [float(v) for v in PRECOMPUTED_VECTORS[w]], 'precomputed'
      if wn in norm_map:
          key = norm_map[wn]
          return [float(v) for v in PRECOMPUTED_VECTORS[key]], 'precomputed'
      if GEMMA_API_ATIVO:
          try:
              resp = gemma_model.generate_content(
                  IAP_PROMPT(w),
                  generation_config={'temperature': 0.1, 'max_output_tokens': 256}
              )
              vec = parse_response(resp.text)
              if vec:
                  return vec, 'gemma-4-31b-it'
          except Exception as e:
              print(f'  [API] Erro para \'{w}\': {e}')
      seed = sum(ord(c) * (i + 1) for i, c in enumerate(w)) % 10000
      rng = np.random.default_rng(seed)
      return [float(round(rng.uniform(2, 8))) for _ in range(N_DIM)], 'hash-fallback'

  print('[VETORES] Computando vetores 12D IAP para 25 pictogramas...')
  print(f'          Modo: {"API Gemma 4 31B" if GEMMA_API_ATIVO else "pre-computados (Gemma 4 31B)"}')

  vectors, sources = [], []
  for k, pic in enumerate(PICTOGRAMAS_IAP):
      vec, src = get_vector(pic)
      vectors.append(vec)
      sources.append(src)
      if (k + 1) % 8 == 0 or k + 1 == len(PICTOGRAMAS_IAP):
          print(f'  {k+1:2d}/{len(PICTOGRAMAS_IAP)} processados')
      if GEMMA_API_ATIVO and src == 'gemma-4-31b-it':
          time.sleep(1.0)

  VECTORS = np.array(vectors, dtype=np.float32)
  src_count = Counter(sources)
  print()
  for s, n in src_count.most_common():
      print(f'  {s:35s}: {n} pictogramas')
  print(f'\n[OK] VECTORS.shape = {VECTORS.shape}')

  print('\nPerfil IAP de 5 pictogramas representativos:')
  for ex_word in ['fome', 'feliz', 'casa', 'quero', 'medico']:
      idx = next((i for i, p in enumerate(PICTOGRAMAS_IAP) if p['palavra'] == ex_word), None)
      if idx is not None:
          v = VECTORS[idx]
          top3 = sorted(zip(IAP_DIMENSOES, v), key=lambda x: -x[1])[:3]
          top3_str = ' | '.join(f'{d[:9]}:{int(s)}' for d, s in top3)
          print(f'  [{PICTOGRAMAS_IAP[idx]["categoria"]:12s}] {ex_word:12s} top3: {top3_str}')

In [ ]:
# =============================================================================
  # CÉLULA 6 — DIAGRAMAS DE PERSISTÊNCIA H0 (Python puro + numpy)
  # Algoritmo:
  #   - Trata o vetor 12D como 12 pontos em R¹
  #   - Ordena os valores: v1 ≤ v2 ≤ ... ≤ v12
  #   - Gaps entre valores adjacentes = filtração em que componentes se fundem
  #   - H0: n-1 barras (nascimento=0, morte=gap)
  # =============================================================================

  def compute_h0_diagram(values):
      v = np.sort(np.array(values, dtype=float))
      gaps = np.diff(v)
      return np.column_stack([np.zeros(len(gaps)), gaps])

  print('[PERSISTENCE] Computando diagramas H0 para 25 pictogramas...')
  diagrams = [compute_h0_diagram(VECTORS[i]) for i in range(len(PICTOGRAMAS_IAP))]
  print(f'[OK] {len(diagrams)} diagramas H0 | {diagrams[0].shape[0]} barras finitas cada')

  exemplos_idx = [
      next(i for i, p in enumerate(PICTOGRAMAS_IAP) if p['palavra'] == 'fome'),
      next(i for i, p in enumerate(PICTOGRAMAS_IAP) if p['palavra'] == 'feliz'),
      next(i for i, p in enumerate(PICTOGRAMAS_IAP) if p['palavra'] == 'quero'),
      next(i for i, p in enumerate(PICTOGRAMAS_IAP) if p['palavra'] == 'casa'),
  ]

  fig, axes = plt.subplots(1, 4, figsize=(16, 4))
  fig.patch.set_facecolor('#1a1a2e')
  fig.suptitle(
      'Diagramas de Persistência H0 — Algoritmo JP (Python puro)\n'
      'Eixo Y = gap (morte − nascimento) = persistência topológica do componente',
      color='white', fontsize=11
  )

  for ax, idx in zip(axes, exemplos_idx):
      ic = PICTOGRAMAS_IAP[idx]
      dgm = diagrams[idx]
      ax.set_facecolor('#0d1117')
      births, deaths = dgm[:, 0], dgm[:, 1]
      ax.scatter(births, deaths, c=CAT_COLORS[ic['categoria']], s=60,
                 zorder=3, alpha=0.9, edgecolors='white', linewidths=0.4)
      lim = max(float(deaths.max()) * 1.1, 0.5)
      ax.plot([0, lim], [0, lim], '--', color='#888', lw=1, alpha=0.5)
      ax.set_xlim(-0.1, 0.2)
      ax.set_ylim(-0.1, lim * 1.05)
      ax.set_xlabel('Nascimento', color='white', fontsize=8)
      ax.set_ylabel('Morte (gap)', color='white', fontsize=8)
      ax.tick_params(colors='white', labelsize=7)
      for spine in ax.spines.values():
          spine.set_edgecolor('#444')
      ax.set_title(f'{ic["palavra"]}\n({ic["categoria"]})',
                   color=CAT_COLORS[ic['categoria']], fontsize=10, fontweight='bold')
      max_pi = int(np.argmax(deaths))
      ax.annotate(f'max={deaths[max_pi]:.1f}', (births[max_pi], deaths[max_pi]),
                  xytext=(3, 3), textcoords='offset points',
                  fontsize=6, color='white', alpha=0.8)

  plt.tight_layout()
  plt.show()
  print('[OK] Diagramas exibidos.')

In [ ]:
# =============================================================================
  # CÉLULA 7 — MATRIZ WASSERSTEIN 25×25 (scipy.optimize.linear_sum_assignment)
  # =============================================================================

  def wasserstein_h0(dgm1, dgm2):
      g1 = np.sort(dgm1[:, 1])
      g2 = np.sort(dgm2[:, 1])
      n1, n2 = len(g1), len(g2)
      n = max(n1, n2)
      a = np.append(g1, np.zeros(n - n1))
      b = np.append(g2, np.zeros(n - n2))
      cost = np.abs(a[:, None] - b[None, :])
      row_ind, col_ind = linear_sum_assignment(cost)
      return float(cost[row_ind, col_ind].sum())

  N = len(PICTOGRAMAS_IAP)
  print(f'[WASSERSTEIN] Computando matriz {N}x{N} = {N*N} pares...')

  W = np.zeros((N, N), dtype=np.float32)
  total_pairs = N * (N - 1) // 2
  done = 0
  for i in range(N):
      for j in range(i + 1, N):
          d = wasserstein_h0(diagrams[i], diagrams[j])
          W[i, j] = W[j, i] = d
          done += 1
      if (i + 1) % 6 == 0 or i + 1 == N:
          print(f'  {done}/{total_pairs} pares ({100*done//total_pairs}%)')

  print(f'\n[OK] Matriz Wasserstein: {W.shape}')
  print(f'     Media: {W[W>0].mean():.4f} | Maxima: {W.max():.4f}')

  fig, ax = plt.subplots(figsize=(11, 9))
  fig.patch.set_facecolor('#1a1a2e')
  ax.set_facecolor('#1a1a2e')
  im = ax.imshow(W, cmap='YlOrRd', aspect='auto')
  cbar = plt.colorbar(im, ax=ax, shrink=0.8)
  cbar.set_label('Distância Wasserstein H0', color='white', fontsize=10)
  cbar.ax.yaxis.set_tick_params(color='white')
  plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')

  labels = [p['palavra'] for p in PICTOGRAMAS_IAP]
  ax.set_xticks(range(N)); ax.set_yticks(range(N))
  ax.set_xticklabels(labels, rotation=90, fontsize=8, color='white')
  ax.set_yticklabels(labels, fontsize=8, color='white')

  prev_cat = PICTOGRAMAS_IAP[0]['categoria']
  for idx, p in enumerate(PICTOGRAMAS_IAP):
      if p['categoria'] != prev_cat:
          ax.axhline(idx - 0.5, color='white', lw=0.9, alpha=0.5)
          ax.axvline(idx - 0.5, color='white', lw=0.9, alpha=0.5)
          prev_cat = p['categoria']

  ax.set_title(
      'Matriz Wasserstein 25×25 — Atlas Afasia (Algoritmo JP + Gemma 4 31B)\n'
      'scipy.optimize.linear_sum_assignment | barras H0: birth=0, death=gap',
      color='white', fontsize=12, pad=15
  )
  for spine in ax.spines.values():
      spine.set_edgecolor('#444')
  plt.tight_layout()
  plt.show()

In [ ]:
# =============================================================================
  # CÉLULA 8 — PROJEÇÃO MDS 2D + SCATTER COLORIDO POR CATEGORIA
  # sklearn.manifold.MDS sobre a matriz Wasserstein
  # =============================================================================

  print('[MDS] Projetando 25 pictogramas em 2D via sklearn.manifold.MDS...')
  mds = MDS(n_components=2, dissimilarity='precomputed',
            random_state=42, metric=True, n_init=4, max_iter=500)
  coords_2d = mds.fit_transform(W)

  pca = PCA(n_components=2)
  pca.fit(coords_2d)
  lambda1 = float(pca.explained_variance_ratio_[0])
  lambda2 = float(pca.explained_variance_ratio_[1])

  print(f'[OK] MDS concluído | stress = {mds.stress_:.4f}')
  print(f'     lambda1 = {lambda1*100:.1f}% | lambda2 = {lambda2*100:.1f}%')
  print(f'     Variância explicada 2D = {(lambda1+lambda2)*100:.1f}%')

  fig, ax = plt.subplots(figsize=(13, 9))
  fig.patch.set_facecolor('#1a1a2e')
  ax.set_facecolor('#0d1117')

  for cat in ['necessidades', 'sentimentos', 'lugares', 'pessoas', 'acoes']:
      idx_cat = [i for i, p in enumerate(PICTOGRAMAS_IAP) if p['categoria'] == cat]
      ax.scatter(
          coords_2d[idx_cat, 0], coords_2d[idx_cat, 1],
          c=CAT_COLORS[cat], s=140, label=CAT_LABELS[cat],
          alpha=0.85, edgecolors='white', linewidths=0.6, zorder=3
      )
      for i in idx_cat:
          ax.annotate(
              PICTOGRAMAS_IAP[i]['palavra'],
              (coords_2d[i, 0], coords_2d[i, 1]),
              textcoords='offset points', xytext=(6, 5),
              fontsize=9, color='white', alpha=0.9
          )

  ax.set_xlabel('MDS-1 (Wasserstein)', color='white', fontsize=11)
  ax.set_ylabel('MDS-2 (Wasserstein)', color='white', fontsize=11)
  ax.set_title(
      f'Atlas Afasia — MDS 2D (Algoritmo JP + Gemma 4 31B)\n'
      f'l1={lambda1*100:.1f}% l2={lambda2*100:.1f}% | stress={mds.stress_:.4f}',
      color='white', fontsize=12, pad=15
  )
  ax.tick_params(colors='white')
  for spine in ax.spines.values():
      spine.set_edgecolor('#444')
  legend = ax.legend(facecolor='#1a1a2e', labelcolor='white',
                     framealpha=0.8, fontsize=10, title='Categoria', title_fontsize=10)
  legend.get_title().set_color('white')
  plt.tight_layout()
  plt.show()

In [ ]:
# =============================================================================
  # CÉLULA 9 — GRAFO DE VIZINHOS TOPOLÓGICOS (matplotlib puro, sem networkx)
  # =============================================================================

  nearest = []
  for i in range(N):
      dists = W[i].copy()
      dists[i] = np.inf
      j = int(np.argmin(dists))
      nearest.append((i, j, float(W[i, j])))

  edges_set, edges = set(), []
  for i, j, d in nearest:
      key = (min(i, j), max(i, j))
      if key not in edges_set:
          edges_set.add(key)
          edges.append((i, j, d))

  print(f'[GRAFO] {N} nós, {len(edges)} arestas')

  fig, ax = plt.subplots(figsize=(14, 10))
  fig.patch.set_facecolor('#1a1a2e')
  ax.set_facecolor('#0d1117')
  ax.axis('off')

  max_d = max(d for _, _, d in edges) if edges else 1.0
  for i, j, d in edges:
      xi, yi = coords_2d[i]
      xj, yj = coords_2d[j]
      alpha_val = 0.3 + 0.5 * (1 - d / max_d)
      ax.plot([xi, xj], [yi, yj], color='#4a90d9', lw=1.8, alpha=alpha_val, zorder=1)

  x_range = coords_2d[:, 0].max() - coords_2d[:, 0].min() or 1.0
  y_range = coords_2d[:, 1].max() - coords_2d[:, 1].min() or 1.0
  dot_r = min(x_range, y_range) * 0.018

  for i, pic in enumerate(PICTOGRAMAS_IAP):
      x, y = coords_2d[i]
      circle = plt.Circle((x, y), dot_r, color=CAT_COLORS[pic['categoria']],
                           zorder=3, alpha=0.9)
      ax.add_patch(circle)
      ax.text(x, y + dot_r * 1.5, pic['palavra'],
              ha='center', va='bottom', fontsize=8,
              color='white', fontweight='bold', zorder=4)

  ax.autoscale()
  ax.set_title(
      'Grafo de Vizinhos Topológicos — Atlas Afasia\n'
      'Cada pictograma conectado ao vizinho Wasserstein mais próximo',
      color='white', fontsize=13, pad=15
  )
  handles = [mpatches.Patch(facecolor=CAT_COLORS[c], label=CAT_LABELS[c])
             for c in ['necessidades', 'sentimentos', 'lugares', 'pessoas', 'acoes']]
  ax.legend(handles=handles, loc='lower right', facecolor='#1a1a2e',
            labelcolor='white', framealpha=0.8, fontsize=9)
  plt.tight_layout()
  plt.show()

  print('\nVizinhos topológicos mais próximos:')
  for i, j, d in nearest:
      pi, pj = PICTOGRAMAS_IAP[i], PICTOGRAMAS_IAP[j]
      print(f'  {pi["palavra"]:12s} ({pi["categoria"]:14s}) -> {pj["palavra"]:12s} ({pj["categoria"]:14s})  d={d:.4f}')

In [ ]:
# =============================================================================
  # CÉLULA 10 — ALGORITMO JP: PLANEJAMENTO REGRESSIVO COM DIJKSTRA
  # G ≈ I + F:
  #   G = caminho topológico ótimo (fome → maca)
  #   I = estado atual do usuário (pictograma inicial)
  #   F = 3 vizinhos de menor custo Wasserstein (próximos passos disponíveis)
  # =============================================================================

  @dataclass
  class PassoPlano:
      numero_passo: int
      de_pictograma: str
      para_pictograma: str
      distancia_wasserstein: float
      distancia_acumulada: float

  @dataclass
  class ResultadoJP:
      sucesso: bool
      caminho: List[PassoPlano]
      custo_total: float
      iteracoes: int
      mensagem: str
      vizinhos_imediatos: List[Dict]

  def dijkstra_jp(idx_inicio, idx_objetivo):
      n = len(PICTOGRAMAS_IAP)
      distancias = [float('inf')] * n
      distancias[idx_inicio] = 0.0
      predecessores = [None] * n
      heap = [(0.0, idx_inicio)]
      visitados = set()

      while heap:
          d, u = heapq.heappop(heap)
          if u in visitados:
              continue
          visitados.add(u)
          if u == idx_objetivo:
              break
          for v in range(n):
              if v in visitados:
                  continue
              novo = d + float(W[u, v])
              if novo < distancias[v]:
                  distancias[v] = novo
                  predecessores[v] = u
                  heapq.heappush(heap, (novo, v))

      caminho = []
      atual = idx_objetivo
      while atual is not None and atual != idx_inicio:
          prev = predecessores[atual]
          if prev is None:
              break
          caminho.append(PassoPlano(
              len(caminho) + 1,
              PICTOGRAMAS_IAP[prev]['palavra'],
              PICTOGRAMAS_IAP[atual]['palavra'],
              float(W[prev, atual]),
              distancias[atual]
          ))
          atual = prev
      caminho.reverse()

      vizinhos = sorted(
          [(float(W[idx_objetivo, j]), j) for j in range(n) if j != idx_objetivo]
      )[:3]

      return ResultadoJP(
          sucesso=bool(caminho),
          caminho=caminho,
          custo_total=distancias[idx_objetivo],
          iteracoes=len(visitados),
          mensagem='Sucesso' if caminho else 'Sem caminho',
          vizinhos_imediatos=[
              {'palavra': PICTOGRAMAS_IAP[j]['palavra'], 'distancia': round(d, 4)}
              for d, j in vizinhos
          ]
      )

  def resolver_picto(nome):
      n = _norm(nome)
      return next((i for i, p in enumerate(PICTOGRAMAS_IAP) if _norm(p['palavra']) == n), None)

  # ── Demo 1: fome → maca ───────────────────────────────────────────────────────
  print('=' * 60)
  print('ALGORITMO JP — DEMO 1: fome → maca')
  print('=' * 60)
  i_fome = resolver_picto('fome')
  i_maca = resolver_picto('maca')
  res1 = dijkstra_jp(i_fome, i_maca)

  if res1.sucesso:
      cam_str = ' -> '.join([res1.caminho[0].de_pictograma] +
                             [p.para_pictograma for p in res1.caminho])
      print(f'Plano: {cam_str}')
      print(f'Custo topológico total: {res1.custo_total:.4f}')
      print(f'Iterações Dijkstra: {res1.iteracoes}')
      print('Passos:')
      for p in res1.caminho:
          print(f'  Passo {p.numero_passo}: {p.de_pictograma:12s} -> {p.para_pictograma:12s}'
                f'  delta={p.distancia_wasserstein:.4f}  sigma={p.distancia_acumulada:.4f}')
      print('Próximos pictogramas disponíveis (F — Flexibilidade):')
      for v in res1.vizinhos_imediatos:
          print(f'  * {v["palavra"]:14s} -> distância {v["distancia"]:6.4f}')

  # ── Demo 2: comer → ajuda ────────────────────────────────────────────────────
  print()
  print('=' * 60)
  print('ALGORITMO JP — DEMO 2: comer → ajuda')
  print('=' * 60)
  i_comer = resolver_picto('comer')
  i_ajuda = resolver_picto('ajuda')
  res2 = dijkstra_jp(i_comer, i_ajuda)

  if res2.sucesso:
      cam_str2 = ' -> '.join([res2.caminho[0].de_pictograma] +
                               [p.para_pictograma for p in res2.caminho])
      print(f'Plano: {cam_str2}')
      print(f'Custo topológico total: {res2.custo_total:.4f}')

  # ── Visualização do caminho ────────────────────────────────────────────────────
  fig, ax = plt.subplots(figsize=(13, 8))
  fig.patch.set_facecolor('#1a1a2e')
  ax.set_facecolor('#0d1117')

  for cat in ['necessidades', 'sentimentos', 'lugares', 'pessoas', 'acoes']:
      idx_c = [i for i, p in enumerate(PICTOGRAMAS_IAP) if p['categoria'] == cat]
      ax.scatter(coords_2d[idx_c, 0], coords_2d[idx_c, 1],
                 c=CAT_COLORS[cat], s=80, alpha=0.4, zorder=2)

  for idx_p, p in enumerate(PICTOGRAMAS_IAP):
      ax.text(coords_2d[idx_p, 0], coords_2d[idx_p, 1] + 0.010,
              p['palavra'], ha='center', fontsize=7, color='#888', zorder=3)

  if res1.sucesso:
      passos_idx = ([resolver_picto(res1.caminho[0].de_pictograma)] +
                    [resolver_picto(p.para_pictograma) for p in res1.caminho])
      cam_x = [coords_2d[i, 0] for i in passos_idx]
      cam_y = [coords_2d[i, 1] for i in passos_idx]
      ax.plot(cam_x, cam_y, 'o-', color='#f1c40f', lw=3, zorder=5, markersize=14,
              markeredgecolor='white', markeredgewidth=2,
              label=f'fome -> maca (custo={res1.custo_total:.3f})')
      for i_step in passos_idx:
          ax.text(coords_2d[i_step, 0], coords_2d[i_step, 1] + 0.018,
                  PICTOGRAMAS_IAP[i_step]['palavra'],
                  ha='center', fontsize=10, color='#f1c40f',
                  fontweight='bold', zorder=6)

  ax.set_title(
      'Algoritmo JP — Caminho Topológico Ótimo (Dijkstra Regressivo)\n'
      'G ≈ I + F: G=caminho | I=fome (estado atual) | F=próximos 3 vizinhos',
      color='white', fontsize=12, pad=15
  )
  ax.set_xlabel('MDS-1', color='white')
  ax.set_ylabel('MDS-2', color='white')
  ax.tick_params(colors='white')
  for spine in ax.spines.values():
      spine.set_edgecolor('#444')
  legend = ax.legend(facecolor='#1a1a2e', labelcolor='white', fontsize=9, loc='upper right')
  plt.tight_layout()
  plt.show()

In [ ]:
# =============================================================================
  # CÉLULA 11 — MÉTRICAS FINAIS + RADAR IAP POR CATEGORIA
  # =============================================================================

  print('=' * 72)
  print('  MÉTRICAS DO ATLAS AFASIA — ALGORITMO JP + GEMMA 4 31B')
  print('=' * 72)
  w_nz = W[W > 0]
  print(f'  Pictogramas total:        {N}')
  print(f'  Dimensoes IAP:            {N_DIM}')
  print(f'  Modelo vetorial:          {GEMMA_MODEL_NAME}')
  print(f'  Metodo de distancia:      Wasserstein H0 (scipy linear_sum_assignment)')
  print(f'  Pipeline:                 100% Python puro (sem dependencias extras)')
  print()
  print(f'  Wasserstein medio:        {w_nz.mean():.4f}')
  print(f'  Wasserstein maximo:       {W.max():.4f}')
  print(f'  Wasserstein minimo:       {w_nz.min():.4f}')
  print(f'  Desvio padrao:            {w_nz.std():.4f}')
  print()
  print(f'  MDS lambda1:              {lambda1*100:.2f}%')
  print(f'  MDS lambda2:              {lambda2*100:.2f}%')
  print(f'  Variancia explicada 2D:   {(lambda1+lambda2)*100:.2f}%')
  print(f'  Stress MDS:               {mds.stress_:.4f}')
  print()
  print('  COESAO POR CATEGORIA (distância Wasserstein intra-categoria):')
  print('  ' + '-' * 60)
  for cat in ['necessidades', 'sentimentos', 'lugares', 'pessoas', 'acoes']:
      idx_c = [i for i, p in enumerate(PICTOGRAMAS_IAP) if p['categoria'] == cat]
      intra = [W[i, j] for ii, i in enumerate(idx_c) for j in idx_c[ii + 1:]]
      coesao = float(np.mean(intra)) if intra else 0.0
      bar = chr(9608) * max(1, int(coesao * 12))
      print(f'  {cat:14s} ({len(idx_c)} pic): {coesao:.4f}  {bar}')
  print('=' * 72)

  fig, (ax1, _) = plt.subplots(1, 2, figsize=(14, 5))
  fig.patch.set_facecolor('#1a1a2e')

  ax1.set_facecolor('#0d1117')
  flat_w = W[np.triu_indices(N, k=1)]
  ax1.hist(flat_w, bins=20, color='#3498db', edgecolor='#1a1a2e', alpha=0.85)
  ax1.axvline(float(flat_w.mean()), color='#e74c3c', lw=2,
              label=f'média={flat_w.mean():.2f}')
  ax1.set_xlabel('Distância Wasserstein', color='white', fontsize=10)
  ax1.set_ylabel('Frequência', color='white', fontsize=10)
  ax1.set_title('Distribuição das Distâncias Wasserstein H0', color='white', fontsize=11)
  ax1.tick_params(colors='white')
  for spine in ax1.spines.values():
      spine.set_edgecolor('#444')
  ax1.legend(fontsize=9, facecolor='#1a1a2e', labelcolor='white')

  cats_r = ['necessidades', 'sentimentos', 'lugares', 'pessoas', 'acoes']
  cat_means = {
      cat: VECTORS[[i for i, p in enumerate(PICTOGRAMAS_IAP) if p['categoria'] == cat]].mean(axis=0)
      for cat in cats_r
  }
  angles = np.linspace(0, 2 * np.pi, N_DIM, endpoint=False).tolist()
  angles += angles[:1]

  ax2 = plt.subplot(122, polar=True)
  ax2.set_facecolor('#0d1117')
  for cat in cats_r:
      vals = cat_means[cat].tolist() + [cat_means[cat][0]]
      ax2.plot(angles, vals, color=CAT_COLORS[cat], lw=2.2, label=CAT_LABELS[cat])
      ax2.fill(angles, vals, color=CAT_COLORS[cat], alpha=0.12)
  ax2.set_xticks(angles[:-1])
  ax2.set_xticklabels([d[:7] for d in IAP_DIMENSOES], color='white', fontsize=7.5)
  ax2.yaxis.set_tick_params(labelcolor='white', labelsize=6)
  ax2.set_ylim(0, 11)
  ax2.spines['polar'].set_color('#444')
  ax2.grid(color='#444', alpha=0.5)
  ax2.set_title('Perfil IAP por Categoria\n(média Gemma 4 31B)',
                color='white', fontsize=11, pad=22)
  ax2.legend(loc='upper right', bbox_to_anchor=(1.5, 1.2),
             facecolor='#1a1a2e', labelcolor='white', framealpha=0.8, fontsize=8)

  fig.suptitle('Métricas Finais — Atlas Afasia (Algoritmo JP + Gemma 4 31B)',
               color='white', fontsize=13, y=1.01)
  plt.tight_layout()
  plt.show()

In [ ]:
# =============================================================================
  # CÉLULA 12 — EXPORTAR atlas_data.json
  # Exporta no formato exato do backend TypeScript IAP
  # =============================================================================

  print('[EXPORT] Gerando atlas_data.json...')

  pictos_export = []
  for i, pic in enumerate(PICTOGRAMAS_IAP):
      x, y = float(coords_2d[i, 0]), float(coords_2d[i, 1])
      dists_i = [(float(W[i, j]), j) for j in range(N) if j != i]
      dists_i.sort()
      vizinhos = [
          {'id': PICTOGRAMAS_IAP[j]['id'],
           'palavra': PICTOGRAMAS_IAP[j]['palavra'],
           'distancia': round(d, 4)}
          for d, j in dists_i[:3]
      ]
      pictos_export.append({
          'id': pic['id'],
          'palavra': pic['palavra'],
          'imagemUrl': pic['url'],
          'categoria': pic['categoria'],
          'coordX': round(x, 3),
          'coordY': round(y, 3),
          'vizinhos': vizinhos,
          'vector': [round(float(v), 2) for v in VECTORS[i]],
      })

  atlas = {
      'pictos': pictos_export,
      'keyDimensions': {
          'lambda1': round(lambda1, 4),
          'lambda2': round(lambda2, 4),
          'varianceExplained': round(lambda1 + lambda2, 4),
      },
      'stats': {
          'totalPictos': N,
          'wasserstein': {
              'mean': round(float(W[W > 0].mean()), 4),
              'max': round(float(W.max()), 4),
              'std': round(float(W[W > 0].std()), 4),
          },
          'mdsStress': round(float(mds.stress_), 4),
          'model': GEMMA_MODEL_NAME,
          'algorithm': 'JP (Passos 2024)',
      }
  }

  with open('atlas_data.json', 'w', encoding='utf-8') as f:
      json.dump(atlas, f, ensure_ascii=False, indent=2)

  print(f'[OK] atlas_data.json exportado: {len(pictos_export)} pictogramas')
  print(f'     Formato: {{pictos, keyDimensions, stats}}')
  print(f'     Destino sugerido: artifacts/api-server/data/afasia_atlas_data.json')

In [ ]:
# =============================================================================
# CÉLULA 13 — ATLAS IAP NOUN PROJECT (120 ícones reais)
# Demonstra a escalabilidade do Algoritmo JP: 3.443 ícones no Atlas completo
# =============================================================================

import json as _json
import numpy as _np
import matplotlib.pyplot as _plt
import matplotlib.patches as _mpatches

NOUN_SAMPLE = _json.loads(r'[{"id":"8195788","palavra":"speech","categoria":"comunicacao","coordX":0.697,"coordY":-0.002},{"id":"8193964","palavra":"speech","categoria":"comunicacao","coordX":0.697,"coordY":-0.001},{"id":"8193930","palavra":"speech","categoria":"comunicacao","coordX":0.702,"coordY":-0.002},{"id":"8186717","palavra":"speech","categoria":"comunicacao","coordX":0.7,"coordY":0.002},{"id":"8186713","palavra":"speech","categoria":"comunicacao","coordX":0.697,"coordY":0.001},{"id":"8186003","palavra":"speech","categoria":"comunicacao","coordX":0.698,"coordY":0.002},{"id":"8181148","palavra":"speech","categoria":"comunicacao","coordX":0.697,"coordY":0.001},{"id":"8177014","palavra":"speech","categoria":"comunicacao","coordX":0.699,"coordY":0},{"id":"8170583","palavra":"speech","categoria":"comunicacao","coordX":0.698,"coordY":0.001},{"id":"8152889","palavra":"speech","categoria":"comunicacao","coordX":0.697,"coordY":0.002},{"id":"8235493","palavra":"phone call","categoria":"emocao","coordX":0.605,"coordY":0.347},{"id":"8205041","palavra":"angry","categoria":"emocao","coordX":0.607,"coordY":0.352},{"id":"199415","palavra":"angry","categoria":"emocao","coordX":0.608,"coordY":0.351},{"id":"8336174","palavra":"angry","categoria":"emocao","coordX":0.603,"coordY":0.347},{"id":"8336168","palavra":"angry","categoria":"emocao","coordX":0.604,"coordY":0.352},{"id":"8325794","palavra":"angry","categoria":"emocao","coordX":0.603,"coordY":0.351},{"id":"8261478","palavra":"angry","categoria":"emocao","coordX":0.605,"coordY":0.348},{"id":"8255703","palavra":"angry","categoria":"emocao","coordX":0.604,"coordY":0.352},{"id":"8248666","palavra":"angry","categoria":"emocao","coordX":0.607,"coordY":0.353},{"id":"8235529","palavra":"angry","categoria":"emocao","coordX":0.603,"coordY":0.348},{"id":"8209343","palavra":"phone call","categoria":"corpo","coordX":0.349,"coordY":0.607},{"id":"8209321","palavra":"phone call","categoria":"corpo","coordX":0.353,"coordY":0.608},{"id":"2952744","palavra":"body","categoria":"corpo","coordX":0.347,"coordY":0.608},{"id":"2619751","palavra":"body","categoria":"corpo","coordX":0.351,"coordY":0.607},{"id":"8197943","palavra":"body","categoria":"corpo","coordX":0.349,"coordY":0.607},{"id":"8187629","palavra":"body","categoria":"corpo","coordX":0.348,"coordY":0.607},{"id":"8171272","palavra":"body","categoria":"corpo","coordX":0.35,"coordY":0.608},{"id":"8165227","palavra":"body","categoria":"corpo","coordX":0.349,"coordY":0.605},{"id":"8165108","palavra":"body","categoria":"corpo","coordX":0.349,"coordY":0.607},{"id":"8129311","palavra":"body","categoria":"corpo","coordX":0.351,"coordY":0.606},{"id":"8232028","palavra":"no food","categoria":"alimentacao","coordX":0.001,"coordY":0.703},{"id":"8228221","palavra":"Food","categoria":"alimentacao","coordX":0.004,"coordY":0.701},{"id":"8228165","palavra":"Food","categoria":"alimentacao","coordX":0.003,"coordY":0.701},{"id":"8221343","palavra":"no food","categoria":"alimentacao","coordX":-0.001,"coordY":0.702},{"id":"8217926","palavra":"Food","categoria":"alimentacao","coordX":0.001,"coordY":0.697},{"id":"8213899","palavra":"Food","categoria":"alimentacao","coordX":0,"coordY":0.701},{"id":"8213894","palavra":"Food","categoria":"alimentacao","coordX":-0.003,"coordY":0.699},{"id":"8212949","palavra":"Food","categoria":"alimentacao","coordX":-0.001,"coordY":0.703},{"id":"8208589","palavra":"Food","categoria":"alimentacao","coordX":-0.003,"coordY":0.699},{"id":"8208479","palavra":"no food","categoria":"alimentacao","coordX":-0.002,"coordY":0.703},{"id":"6187604","palavra":"mother","categoria":"familia_pessoas","coordX":-0.354,"coordY":0.607},{"id":"6187597","palavra":"mother","categoria":"familia_pessoas","coordX":-0.344,"coordY":0.607},{"id":"6163036","palavra":"mother","categoria":"familia_pessoas","coordX":-0.349,"coordY":0.607},{"id":"6145754","palavra":"mother","categoria":"familia_pessoas","coordX":-0.354,"coordY":0.605},{"id":"6071657","palavra":"mother","categoria":"familia_pessoas","coordX":-0.351,"coordY":0.607},{"id":"6019879","palavra":"mother","categoria":"familia_pessoas","coordX":-0.354,"coordY":0.607},{"id":"6019878","palavra":"mother","categoria":"familia_pessoas","coordX":-0.353,"coordY":0.608},{"id":"6019876","palavra":"mother","categoria":"familia_pessoas","coordX":-0.35,"coordY":0.602},{"id":"6019871","palavra":"mother","categoria":"familia_pessoas","coordX":-0.348,"coordY":0.606},{"id":"6019855","palavra":"mother","categoria":"familia_pessoas","coordX":-0.348,"coordY":0.602},{"id":"6815413","palavra":"people","categoria":"acao","coordX":-0.609,"coordY":0.351},{"id":"6815390","palavra":"people","categoria":"acao","coordX":-0.604,"coordY":0.352},{"id":"35891","palavra":"people","categoria":"acao","coordX":-0.602,"coordY":0.35},{"id":"8325226","palavra":"people","categoria":"acao","coordX":-0.605,"coordY":0.353},{"id":"8312543","palavra":"people","categoria":"acao","coordX":-0.607,"coordY":0.349},{"id":"7437871","palavra":"run","categoria":"acao","coordX":-0.604,"coordY":0.349},{"id":"7417913","palavra":"run","categoria":"acao","coordX":-0.609,"coordY":0.348},{"id":"7417815","palavra":"run","categoria":"acao","coordX":-0.604,"coordY":0.35},{"id":"7412326","palavra":"run","categoria":"acao","coordX":-0.606,"coordY":0.353},{"id":"7412248","palavra":"run","categoria":"acao","coordX":-0.609,"coordY":0.35},{"id":"8294241","palavra":"phone call","categoria":"lugar","coordX":-0.7,"coordY":-0.003},{"id":"8294219","palavra":"phone call","categoria":"lugar","coordX":-0.703,"coordY":-0.002},{"id":"8278796","palavra":"phone call","categoria":"lugar","coordX":-0.698,"coordY":-0.002},{"id":"8278785","palavra":"phone call","categoria":"lugar","coordX":-0.697,"coordY":0},{"id":"8337042","palavra":"Hospital","categoria":"lugar","coordX":-0.697,"coordY":0.003},{"id":"8322295","palavra":"Hospital","categoria":"lugar","coordX":-0.697,"coordY":0},{"id":"8322271","palavra":"Hospital","categoria":"lugar","coordX":-0.699,"coordY":0.002},{"id":"8219947","palavra":"Hospital","categoria":"lugar","coordX":-0.7,"coordY":-0.002},{"id":"8095337","palavra":"Hospital","categoria":"lugar","coordX":-0.701,"coordY":0.002},{"id":"7841529","palavra":"Hospital","categoria":"lugar","coordX":-0.703,"coordY":0.001},{"id":"8172294","palavra":"sick","categoria":"saude","coordX":-0.612,"coordY":-0.352},{"id":"8118425","palavra":"sick","categoria":"saude","coordX":-0.601,"coordY":-0.35},{"id":"8118390","palavra":"sick","categoria":"saude","coordX":-0.602,"coordY":-0.355},{"id":"8005987","palavra":"sick","categoria":"saude","coordX":-0.608,"coordY":-0.354},{"id":"8005974","palavra":"sick","categoria":"saude","coordX":-0.613,"coordY":-0.347},{"id":"7686643","palavra":"sick","categoria":"saude","coordX":-0.611,"coordY":-0.347},{"id":"7621637","palavra":"sick","categoria":"saude","coordX":-0.607,"coordY":-0.347},{"id":"7618096","palavra":"sick","categoria":"saude","coordX":-0.604,"coordY":-0.356},{"id":"7618091","palavra":"sick","categoria":"saude","coordX":-0.598,"coordY":-0.346},{"id":"7456058","palavra":"sick","categoria":"saude","coordX":-0.609,"coordY":-0.346},{"id":"8196764","palavra":"call phone","categoria":"natureza","coordX":-0.354,"coordY":-0.604},{"id":"8219466","palavra":"Rain","categoria":"natureza","coordX":-0.349,"coordY":-0.61},{"id":"8188113","palavra":"raining","categoria":"natureza","coordX":-0.353,"coordY":-0.607},{"id":"8181439","palavra":"Rain","categoria":"natureza","coordX":-0.354,"coordY":-0.603},{"id":"8153927","palavra":"Rain","categoria":"natureza","coordX":-0.351,"coordY":-0.608},{"id":"8139605","palavra":"Rain","categoria":"natureza","coordX":-0.346,"coordY":-0.609},{"id":"8139598","palavra":"Rain","categoria":"natureza","coordX":-0.349,"coordY":-0.606},{"id":"8139595","palavra":"Rain","categoria":"natureza","coordX":-0.346,"coordY":-0.605},{"id":"8138487","palavra":"Rain","categoria":"natureza","coordX":-0.35,"coordY":-0.608},{"id":"8138473","palavra":"Rain","categoria":"natureza","coordX":-0.352,"coordY":-0.608},{"id":"8259412","palavra":"Morning","categoria":"tempo","coordX":0,"coordY":-0.704},{"id":"6886551","palavra":"Morning","categoria":"tempo","coordX":-0.002,"coordY":-0.698},{"id":"5991191","palavra":"Morning","categoria":"tempo","coordX":-0.006,"coordY":-0.697},{"id":"5774807","palavra":"Morning","categoria":"tempo","coordX":-0.003,"coordY":-0.697},{"id":"5645522","palavra":"Morning","categoria":"tempo","coordX":0,"coordY":-0.699},{"id":"5367951","palavra":"Morning","categoria":"tempo","coordX":-0.002,"coordY":-0.698},{"id":"4489585","palavra":"Morning","categoria":"tempo","coordX":-0.004,"coordY":-0.7},{"id":"4489551","palavra":"Morning","categoria":"tempo","coordX":-0.001,"coordY":-0.699},{"id":"3441544","palavra":"Morning","categoria":"tempo","coordX":0.005,"coordY":-0.696},{"id":"3441419","palavra":"Morning","categoria":"tempo","coordX":0.005,"coordY":-0.703},{"id":"8332406","palavra":"phone call","categoria":"objeto","coordX":0.351,"coordY":-0.602},{"id":"8082938","palavra":"Phone","categoria":"objeto","coordX":0.347,"coordY":-0.605},{"id":"3612568","palavra":"Phone","categoria":"objeto","coordX":0.347,"coordY":-0.604},{"id":"8358241","palavra":"phone call","categoria":"objeto","coordX":0.351,"coordY":-0.601},{"id":"8333511","palavra":"phone call","categoria":"objeto","coordX":0.352,"coordY":-0.608},{"id":"8332458","palavra":"phone call","categoria":"objeto","coordX":0.347,"coordY":-0.611},{"id":"8325802","palavra":"phone call","categoria":"objeto","coordX":0.35,"coordY":-0.611},{"id":"8309421","palavra":"phone call","categoria":"objeto","coordX":0.357,"coordY":-0.609},{"id":"8271777","palavra":"phone call","categoria":"objeto","coordX":0.353,"coordY":-0.611},{"id":"8263932","palavra":"phone call","categoria":"objeto","coordX":0.349,"coordY":-0.607},{"id":"7070515","palavra":"Class","categoria":"escola","coordX":0.613,"coordY":-0.352},{"id":"7067075","palavra":"Class","categoria":"escola","coordX":0.612,"coordY":-0.359},{"id":"6948050","palavra":"Class","categoria":"escola","coordX":0.614,"coordY":-0.351},{"id":"6684180","palavra":"Class","categoria":"escola","coordX":0.61,"coordY":-0.354},{"id":"6194483","palavra":"Class","categoria":"escola","coordX":0.6,"coordY":-0.351},{"id":"6158535","palavra":"Class","categoria":"escola","coordX":0.624,"coordY":-0.347},{"id":"6158532","palavra":"Class","categoria":"escola","coordX":0.605,"coordY":-0.346},{"id":"8104917","palavra":"classes","categoria":"escola","coordX":0.603,"coordY":-0.354},{"id":"8067960","palavra":"Class","categoria":"escola","coordX":0.614,"coordY":-0.349},{"id":"7954515","palavra":"Class","categoria":"escola","coordX":0.614,"coordY":-0.351}]')

CAT_COLORS_NOUN = {
    'comunicacao': '#3498db', 'emocao': '#9b59b6', 'corpo': '#e74c3c',
    'alimentacao': '#f39c12', 'familia_pessoas': '#1abc9c', 'acao': '#e67e22',
    'lugar': '#27ae60', 'saude': '#c0392b', 'natureza': '#2ecc71',
    'tempo': '#95a5a6', 'objeto': '#7f8c8d', 'escola': '#d35400',
}

xs = _np.array([d['coordX'] for d in NOUN_SAMPLE], dtype=float)
ys = _np.array([d['coordY'] for d in NOUN_SAMPLE], dtype=float)
cats = [d['categoria'] for d in NOUN_SAMPLE]
unique_cats = sorted(set(cats))

fig, ax = _plt.subplots(figsize=(13, 9))
fig.patch.set_facecolor('#1a1a2e')
ax.set_facecolor('#0d1117')

for cat in unique_cats:
    mask = [i for i, c in enumerate(cats) if c == cat]
    color = CAT_COLORS_NOUN.get(cat, '#888888')
    ax.scatter(xs[mask], ys[mask], c=color, s=30, alpha=0.7, label=cat,
               edgecolors='none', zorder=3)

ax.set_xlabel('MDS-1 (Wasserstein)', color='white', fontsize=11)
ax.set_ylabel('MDS-2 (Wasserstein)', color='white', fontsize=11)
ax.set_title(
    f'Atlas Noun Project — Amostra de {len(NOUN_SAMPLE)} / 3.443 ícones\\n'
    'Algoritmo JP + Gemma 4 31B | Mesma pipeline topológica do Atlas Afasia',
    color='white', fontsize=12, pad=15
)
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_edgecolor('#444')
legend = ax.legend(facecolor='#1a1a2e', labelcolor='white', framealpha=0.8,
                   fontsize=8, title='Categoria', title_fontsize=9,
                   ncol=2, loc='upper right')
legend.get_title().set_color('white')
_plt.tight_layout()
_plt.show()

cat_count = {}
for d in NOUN_SAMPLE:
    cat_count[d['categoria']] = cat_count.get(d['categoria'], 0) + 1
print(f'\\nDistribuição nas {len(unique_cats)} categorias da amostra ({len(NOUN_SAMPLE)} ícones):')
for cat, n in sorted(cat_count.items(), key=lambda x: -x[1]):
    print(f'  {cat:20s}: {n:3d}')


## Conclusão

  ### O que demonstramos

  Este notebook aplicou o **Algoritmo JP** (Inteligência Artificial Pictórica, Passos 2024) ao domínio da **afasia** em quatro etapas, usando apenas bibliotecas padrão Python:

  1. **Fase Semântica (Gemma 4 31B via API)** — O modelo atribuiu vetores 12D aos 25 pictogramas AFASIA, capturando nuances que métodos clássicos não conseguem.

  2. **Fase Topológica (Python puro)** — Diagramas de persistência H0 via numpy (gaps entre valores ordenados). Distância Wasserstein via `scipy.optimize.linear_sum_assignment`.

  3. **Projeção e Grafo** — `sklearn.manifold.MDS` preservando distâncias topológicas. Grafo de vizinhos desenhado com matplotlib puro.

  4. **Algoritmo JP — Dijkstra Regressivo** — Planejamento semântico ótimo no espaço topológico. A equação G ≈ I + F é operacionalizada: o caminho (G) é encontrado por Dijkstra partindo do estado atual (I), com os 3 vizinhos de menor custo Wasserstein como opções de flexibilidade (F).

  ### Diferença AFASIA vs DISFASIA

  | Aspecto | Atlas AFASIA | Atlas DISFASIA |
  |---------|-------------|----------------|
  | Pictogramas | 25 (ARASAAC, 5 categorias gerais) | 38 (ARASAAC, 5 categorias disfasia) |
  | Algoritmo extra | **Dijkstra regressivo** (planejamento) | Apenas visualização |
  | Público | Adultos pós-AVC | Crianças com disfasia |
  | Atlas externo | Noun Project 3k (3.443 ícones) | — |

  ### Referências

  - Passos, J.P.P. (2024). *Inteligência Artificial Pictórica: Algoritmo JP para Atlas Topológico de Pictogramas CAA*. UFT — Universidade Federal do Tocantins.
  - ARASAAC — Centro Aragonés para la Comunicación Aumentativa y Alternativa. CC BY-NC-SA. https://arasaac.org
  - Google DeepMind. (2025). *Gemma 4: Open Models Based on Gemini Research and Technology*.
  - Edelsbrunner, H., Letscher, D., Zomorodian, A. (2002). *Topological Persistence and Simplification*. Discrete & Computational Geometry.
  - Villani, C. (2009). *Optimal Transport: Old and New*. Springer.